# 04 — Baselines: structured-only and zero-shot LLM

The two baselines of Section 3.2.4 that need a GPU (#30, #52). The rule-based and
Isolation-Forest baselines run on the CPU locally. Both write predictions in the shared
format (`experiments/predictions/<system>/<split>.csv`) for `src.evaluation.heldout`.

| Step | Module | Output (in Drive) |
|---|---|---|
| Structured-only: export, train 5 seeds, predict | `src.baselines.structured_only`, `src.classification.train` | `experiments/structured_only/<version>/`, `predictions/structured_only_seed<N>/` |
| Zero-shot LLM | `src.baselines.zero_shot_llm run` | `experiments/zero_shot_llm/<version>/`, `predictions/zero_shot_llm/` |

Training needs only the first run (validation). For the final evaluation, set `SPLIT = 'test'`
and `FINAL = True`, set `TRAIN_STRUCTURED = False` (the checkpoints exist) and run again.


In [1]:
DATASET_VERSION = 'v0.6-20k'
BRANCH = 'dev'
SPLIT = 'test'      # 'test' only for the final evaluation, with FINAL = True
FINAL = True
BASE_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'   # the same model family as the classifier
SEEDS = [42, 123, 2024]
EPOCHS = 5
TRAIN_STRUCTURED = False  # False once the five checkpoints exist
RUN_ZERO_SHOT = True
ZERO_SHOT_LIMIT = None   # e.g. 1000 if GPU time runs short (state it when reporting)
LLM_MODEL = 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'
RUN_RETRAINING = False   # True once, on the validation run (#31): needs the active checkpoint and feedback examples


## 1. Bootstrap: Drive, repository, paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_DIR = '/content/claimguard'

DRIVE_ENV = '/content/drive/MyDrive/claimguard/.env'

def load_env_file(path):
    # Minimal .env reader (KEY=value lines), so this cell needs no extra package.
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"\''))

def github_token():
    # The kernel runs on Colab's VM, so a local .env (VS Code's jupyter.envFile) never
    # reaches it. Read the .env kept in Drive, then a Colab secret, then prompt.
    load_env_file(DRIVE_ENV)
    if os.getenv('GITHUB_TOKEN'):
        return os.environ['GITHUB_TOKEN']
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    from getpass import getpass
    return getpass('GITHUB_TOKEN (read-only, not saved): ').strip()

if not os.path.exists(REPO_DIR):
    token = github_token()
    if not token:
        raise RuntimeError('No GITHUB_TOKEN: add a Colab secret or enter it at the prompt')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    f'https://{token}@github.com/victorgpt0/claimguard.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
ROOT = '/content/drive/MyDrive/claimguard'
PROCESSED = f'{ROOT}/data/processed/{DATASET_VERSION}'
CACHE = f'{ROOT}/artifacts/icd11_cache.json'
EXPERIMENTS = f'{ROOT}/experiments/classifier/{DATASET_VERSION}'
PREDICTIONS = f'{ROOT}/experiments/predictions/{DATASET_VERSION}'
STRUCT_EXPORT = '/content/structured_data'   # temporary, rebuilt from PROCESSED
STRUCT_EXPERIMENTS = f'{ROOT}/experiments/structured_only/{DATASET_VERSION}'
ZERO_SHOT_RUNS = f'{ROOT}/experiments/zero_shot_llm/{DATASET_VERSION}'
FINAL_FLAG = '--final' if FINAL else ''
if SPLIT == 'test' and not FINAL:
    raise ValueError('The test split is read once, for the final evaluation: set FINAL = True deliberately')
for path in (f'{PROCESSED}/{SPLIT}.parquet', CACHE):
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path} missing: upload it to Drive first')
for d in (PREDICTIONS, STRUCT_EXPERIMENTS, ZERO_SHOT_RUNS):
    os.makedirs(d, exist_ok=True)
print('commit', COMMIT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
commit 10bb75ce13915b783332bcde541dc1ab38843dab


In [3]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip install -q vllm==0.19.0 peft
# Colab preinstalls torchao 0.10, which this project doesn't use; peft >= 0.21 refuses to
# start LoRA when an old torchao is present ("only versions above 0.16.0 are supported").
!pip uninstall -q -y torchao
import torch
print('cuda', torch.cuda.is_available())


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
cuda True


## 2. Structured-only baseline (#30)

The classifier's model and training, with the structured billing fields in place of the
notes, and `is_fraud` as the label.

In [4]:
!python -m src.baselines.structured_only export --data {PROCESSED} --cache {CACHE} --out {STRUCT_EXPORT}


train: 10800 examples
validation: 3600 examples
test: 3600 examples
deployment_pool: 2000 examples


In [5]:
if TRAIN_STRUCTURED:
    seeds = ' '.join(map(str, SEEDS))
    !python -m src.classification.train --train-file {STRUCT_EXPORT}/train.jsonl --val-file {STRUCT_EXPORT}/validation.jsonl \
        --base-model {BASE_MODEL} --experiments-dir {STRUCT_EXPERIMENTS} --seeds {seeds} --epochs {EPOCHS}


In [6]:
!python -m src.baselines.structured_only predict --runs {STRUCT_EXPERIMENTS} --data {STRUCT_EXPORT} \
    --split {SPLIT} {FINAL_FLAG} --base-model {BASE_MODEL} --predictions-root {PREDICTIONS}
import json, glob
for path in sorted(glob.glob(f'{STRUCT_EXPERIMENTS}/*/config.json')):
    record = json.load(open(path))
    record.setdefault('colab_commit', COMMIT)
    json.dump(record, open(path, 'w'), indent=2)
    m = record['metrics']
    print(f"{record['run_id']}: validation precision {m['precision']:.3f}, recall {m['recall']:.3f}, f1 {m['f1']:.3f}")
    

I0000 00:00:1793136387.883794   12515 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
skipped 20261025T180826_seed42: no metrics.json (stopped run)
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertF

## 3. Zero-shot LLM baseline (#52)

Llama 3.1 8B, temperature 0, one-word answer. The run folder keeps the prompt, every raw
answer and the unparseable count. Putting the answers in the classifier's place in the
funnel (`zero_shot_llm combine`) needs the pipeline's stage-1 predictions for the same
split, so it runs after notebook 03 with `WRITE_PREDICTIONS = True`; it needs no GPU.

In [7]:
if RUN_ZERO_SHOT:
    limit = f'--limit {ZERO_SHOT_LIMIT}' if ZERO_SHOT_LIMIT else ''
    !python -m src.baselines.zero_shot_llm run --data {PROCESSED} --split {SPLIT} {FINAL_FLAG} --cache {CACHE} \
        --out {ZERO_SHOT_RUNS} --model {LLM_MODEL} --predictions-root {PREDICTIONS} {limit}
    import os
    stage1 = f'{PREDICTIONS}/pipeline_stage1/{SPLIT}.csv'
    if os.path.exists(stage1):
        !python -m src.baselines.zero_shot_llm combine --split {SPLIT} {FINAL_FLAG} --predictions-root {PREDICTIONS}
    else:
        print(f'{stage1} not found yet: run notebook 03 on {SPLIT} with WRITE_PREDICTIONS = True, then `combine`.')


I0000 00:00:1793136542.411792   13166 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 10-27 21:29:10 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 2048, 'disable_log_stats': True, 'model': 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'}
INFO 10-27 21:29:11 [model.py:549] Resolved architecture: LlamaForCausalLM
INFO 10-27 21:29:11 [model.py:1678] Using max model len 2048
INFO 10-27 21:29:13 [awq_marlin.py:245] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 10-27 21:29:13 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=2048.
Parse safetensors files: 100% 2/2 [00:00<00:00,  4.74it/s]
INFO 10-27 21:29:13 [vllm.py:790] 

## 4. Feedback retraining run (#31)

Trains one candidate on the training split plus the unconsumed Feedback Dataset examples
(59 simulated ones from 19 Oct) and promotes it to `models/classifier_lora/active/` only if
its F1 and recall on the test split are not worse (FR-14). `evaluated_v1/` is never touched:
Chapter 5 reports that frozen checkpoint. **Close the local dashboard while this runs**: it
writes the same Feedback Dataset and Drive files. Never pass `--simulate-auditor` again.

In [8]:
if RUN_RETRAINING:
    CLASSIFIER_EXPORT = '/content/classifier_data'
    !python -m src.classification.export --data {PROCESSED} --cache {CACHE} --out {CLASSIFIER_EXPORT}
    !python -m src.feedback.run_retraining --config {REPO_DIR}/config/config.example.yaml \
        --classifier-data {CLASSIFIER_EXPORT}
    !tail -n 1 {ROOT}/experiments/retraining_history.jsonl
